# **RQ1 :** Tracking Quality Reliability Evaluation

1. Tracking Coverage Left and Right with raw tracking data vs processed Data
2. report the median gap length and % of interpolated frames


In [1]:
#imports
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

if os.getcwd().endswith("notebooks_final"):
    os.chdir("../")
from src_final.features.global_feature_extractor import SurgicalFeatureExtractor
from src_final.models.analysis import leakage_free_correlation_analysis, leakage_free_residual_analysis
from src_final.models.baseline_loso import evaluate_loso_model

In [13]:
# paths
raw_data_path = "data/raw/output_dataframes/"
processed_data_path = "data/processed/landmark_dataframes/"

raw_files = [f for f in os.listdir(raw_data_path) if '30fps' in f and '2024' in f and f.endswith('.pkl')]
processed_files = [f for f in os.listdir(processed_data_path) if '30fps' in f and '2024' in f and f.endswith('.pkl')]

print(f"Found {len(raw_files)} raw files and {len(processed_files)} processed files.")

Found 86 raw files and 86 processed files.


In [3]:
df = pd.read_pickle(os.path.join(raw_data_path, raw_files[0]))

In [6]:
df

,frame,hand_label,hand_score,bbox_center,palm_center,landmarks
0,0,Left,0.994533,"(760.0, 515.5)","(725.3055000305176, 539.5601069927216)","[{'id': 0, 'coord': (0.3763466477394104, 0.506..."
1,1,None,NaN,"(None, None)","(None, None)",None
2,2,Left,0.898004,"(793.5, 499.5)","(804.667558670044, 510.1365637779236)","[{'id': 0, 'coord': (0.4071502983570099, 0.495..."
3,3,Left,0.966944,"(803.5, 509.0)","(782.1713352203369, 521.3963538408279)","[{'id': 0, 'coord': (0.4027369022369385, 0.509..."
4,4,Left,0.975589,"(814.5, 500.5)","(773.1962966918945, 517.9409122467041)","[{'id': 0, 'coord': (0.39946383237838745, 0.49..."
...,...,...,...,...,...,...
37048,20811,Left,0.988997,"(786.5, 318.0)","(746.447696685791, 305.3677475452423)","[{'id': 0, 'coord': (0.39710527658462524, 0.29..."
37049,20812,Right,0.991073,"(1316.5, 538.0)","(1362.0641899108887, 492.12796568870544)","[{'id': 0, 'coord': (0.7170190215110779, 0.432..."
37050,20812,Left,0.989806,"(780.5, 329.0)","(740.6752395629883, 317.7347695827484)","[{'id': 0, 'coord': (0.39365291595458984, 0.30..."
37051,20813,Right,0.995585,"(1306.5, 539.5)","(1354.2010307312012, 494.2696934938431)","[{'id': 0, 'coord': (0.7142150402069092, 0.436..."


In [11]:
df_raw_right = df[df['hand_label'] == 'Left']

df_raw_right['frame'].nunique() / df['frame'].nunique()
#df_raw_right['frame'].nunique() / (df_raw_right['frame'].max() - df_raw_right['frame'].min() + 1)

0.8055635629864514

In [16]:
raw_coverages_left = []
raw_coverages_right = []

for filename in tqdm(raw_files):
    # Load raw tracking data
    df_raw = pd.read_pickle(os.path.join(raw_data_path, filename))

    # swaped label on purpose (mediapipe gets it wrong)
    df_raw_right = df_raw[df_raw['hand_label'] == 'Left']
    df_raw_left = df_raw[df_raw['hand_label'] == 'Right']

    right_coverage = df_raw_right['frame'].nunique() / df_raw['frame'].nunique()
    left_coverage = df_raw_left['frame'].nunique() / df_raw['frame'].nunique()
    
    raw_coverages_right.append(right_coverage)
    raw_coverages_left.append(left_coverage)

100%|██████████| 86/86 [00:49<00:00,  1.75it/s]


In [ ]:
processed_coverages_left = []
processed_coverages_right = []

fracs_continous_traj_left = []
fracs_continous_traj_right = []

perc_interpolated_left_list = []
perc_interpolated_right_list = []

for filename in tqdm(processed_files):
    # Load processed tracking data
    df_processed = pd.read_pickle(os.path.join(processed_data_path, filename))

    df_processed_right = df_processed[df_processed['hand_label'] == 'Right']
    df_processed_left = df_processed[df_processed['hand_label'] == 'Left']

    right_coverage = df_processed_right['frame'].nunique() / df_processed['frame'].nunique()
    left_coverage = df_processed_left['frame'].nunique() / df_processed['frame'].nunique()
    
    processed_coverages_right.append(right_coverage)
    processed_coverages_left.append(left_coverage)

    # fraction of tracked frames in segments longer than 1.5sec
    seg_lengths_right = df_processed_right.groupby('segment_id').size()
    seg_lengths_left = df_processed_left.groupby('segment_id').size()

    # at least 1.5 sec (45 frames at 30fps)
    frac_right = seg_lengths_right[seg_lengths_right >= 45].sum() / df_processed_right['frame'].nunique()
    frac_left = seg_lengths_left[seg_lengths_left >= 45].sum() / df_processed_left['frame'].nunique()
    fracs_continous_traj_right.append(frac_right)
    fracs_continous_traj_left.append(frac_left)

    # percentage of interpolated frames
    perc_interpolated_right = df_processed_right['hand_score'].isna().sum() / len(df_processed_right)
    perc_interpolated_left = df_processed_left['hand_score'].isna().sum() / len(df_processed_left)
    perc_interpolated_right_list.extend([perc_interpolated_right])
    perc_interpolated_left_list.extend([perc_interpolated_left])

100%|██████████| 86/86 [00:05<00:00, 16.35it/s]


In [34]:
# raw vs processed coverage
print(f"Raw left hand coverage: {np.mean(raw_coverages_left):.2%} ± {np.std(raw_coverages_left):.2%}")
print(f"Raw right hand coverage: {np.mean(raw_coverages_right):.2%} ± {np.std(raw_coverages_right):.2%}")

print(f"Processed left hand coverage: {np.mean(processed_coverages_left):.2%} ± {np.std(processed_coverages_left):.2%}")
print(f"Processed right hand coverage: {np.mean(processed_coverages_right):.2%} ± {np.std(processed_coverages_right):.2%}")

Raw left hand coverage: 81.83% ± 9.31%
Raw right hand coverage: 78.58% ± 6.48%
Processed left hand coverage: 88.71% ± 8.74%
Processed right hand coverage: 84.63% ± 6.04%


In [41]:
# number of interpolated frames
print(f"Percentage of interpolated frames (left hand): {np.mean(perc_interpolated_left_list):.4f} ± {np.std(perc_interpolated_left_list):.4f}")
print(f"Percentage of interpolated frames (right hand): {np.mean(perc_interpolated_right_list):.4f} ± {np.std(perc_interpolated_right_list):.4f}")

Percentage of interpolated frames (left hand): 0.0131 ± 0.0089
Percentage of interpolated frames (right hand): 0.0066 ± 0.0040


In [56]:
# median gap length distribution
print(f"Distribution of continous trajectory fraction (left hand):\n {pd.Series(fracs_continous_traj_left).describe()}")
print(f"Distribution of continous trajectory fraction (right hand):\n {pd.Series(fracs_continous_traj_right).describe()}")

Distribution of continous trajectory fraction (left hand):
 count    86.000000
mean      0.960015
std       0.026410
min       0.878300
25%       0.943555
50%       0.966238
75%       0.978563
max       0.997812
dtype: float64
Distribution of continous trajectory fraction (right hand):
 count    86.000000
mean      0.970329
std       0.016501
min       0.893306
25%       0.964004
50%       0.973129
75%       0.981794
max       0.996821
dtype: float64
